In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
ruta_tiles = "tiles/data/tiles-v2-2021-2024"

In [3]:
meta = pd.read_parquet(ruta_tiles + "/tiles_meta.parquet")

with np.load(ruta_tiles + "/tiles_train.npz") as data:
    tiles = data["data"]
    bands = data["bands"]

In [4]:
print(tiles.shape)
print(bands)

print(meta.shape)
print(meta["clase"].value_counts())

print(meta.isna().mean().sort_values(ascending=False).head(10))

print(meta[["ndvi", "ndbi", "scl_pct", "no2", "so2", "o3"]].describe().round(4))

(1150, 13, 64, 64)
['B1' 'B2' 'B3' 'B4' 'B5' 'B6' 'B7' 'B8' 'B8A' 'B9' 'B11' 'B12' 'SCL']
(1150, 23)
clase
contaminacion_alta_NO2    230
vegetacion_densa          230
ozono_anomalo             230
contaminacion_alta_SO2    230
suelo_urbano              230
Name: count, dtype: int64
no2              0.171304
modis_AOD_055    0.073043
modis_AOD_047    0.073043
o3               0.053913
clase            0.000000
time_s2          0.000000
fecha_s2         0.000000
ndbi             0.000000
ndvi             0.000000
lon              0.000000
dtype: float64
            ndvi       ndbi    scl_pct       no2        so2         o3
count  1150.0000  1150.0000  1150.0000  953.0000  1150.0000  1088.0000
mean      0.5011    -0.1295     0.8748    0.0000     0.0001     0.1188
std       0.2229     0.1647     0.1921    0.0000     0.0002     0.0065
min      -0.0175    -0.4521     0.3015   -0.0000    -0.0007     0.1053
25%       0.2791    -0.2644     0.8104    0.0000     0.0000     0.1142
50%       0.5750

In [5]:
conteo = meta["clase"].value_counts()
print("Balance de clases")
print(conteo)
print(f"Min/Max ratio: {conteo.min()/conteo.max():.4f}")
entropy = -(conteo / conteo.sum() * np.log2(conteo / conteo.sum())).sum()
print(f"Entropia: {entropy:.4f}")
print(f"Max entropy posible: {np.log2(len(conteo)):.4f}")

Balance de clases
clase
contaminacion_alta_NO2    230
vegetacion_densa          230
ozono_anomalo             230
contaminacion_alta_SO2    230
suelo_urbano              230
Name: count, dtype: int64
Min/Max ratio: 1.0000
Entropia: 2.3219
Max entropy posible: 2.3219


In [6]:
num_cols = ["ndvi", "ndbi", "scl_pct", "no2", "so2", "o3", "modis_AOD_047", "modis_AOD_055"]
corr = meta[num_cols].corr()
print("Correlacion")
print(corr.round(3))
c = corr.abs().unstack()
c = c[c > 0.5].dropna()
c = c[c.index.get_level_values(0) < c.index.get_level_values(1)]
if len(c):
    for (i, j), v in c.items():
        print(f"  {i} vs {j}: {v:.3f}")

Correlacion
                ndvi   ndbi  scl_pct    no2    so2     o3  modis_AOD_047  \
ndvi           1.000 -0.912    0.187 -0.415  0.095 -0.033         -0.234   
ndbi          -0.912  1.000    0.039  0.398 -0.094  0.071          0.149   
scl_pct        0.187  0.039    1.000  0.031 -0.014  0.003         -0.080   
no2           -0.415  0.398    0.031  1.000 -0.069 -0.136          0.131   
so2            0.095 -0.094   -0.014 -0.069  1.000  0.011         -0.094   
o3            -0.033  0.071    0.003 -0.136  0.011  1.000         -0.217   
modis_AOD_047 -0.234  0.149   -0.080  0.131 -0.094 -0.217          1.000   
modis_AOD_055 -0.234  0.149   -0.080  0.131 -0.094 -0.217          1.000   

               modis_AOD_055  
ndvi                  -0.234  
ndbi                   0.149  
scl_pct               -0.080  
no2                    0.131  
so2                   -0.094  
o3                    -0.217  
modis_AOD_047          1.000  
modis_AOD_055          1.000  
  ndbi vs ndvi: 0.912
  

In [7]:
print("Stats por clase")
for col in ["ndvi", "ndbi", "scl_pct", "no2", "so2", "o3", "modis_AOD_047", "modis_AOD_055"]:
    print(f"\n{col}")
    print(meta.groupby("clase")[col].describe().round(4))

Stats por clase

ndvi
                        count    mean     std     min     25%     50%     75%  \
clase                                                                           
contaminacion_alta_NO2  230.0  0.4942  0.1830  0.0327  0.3501  0.5005  0.6450   
contaminacion_alta_SO2  230.0  0.6124  0.1542  0.1341  0.5253  0.6563  0.7254   
ozono_anomalo           230.0  0.5268  0.1681  0.1196  0.4113  0.5492  0.6504   
suelo_urbano            230.0  0.1777  0.0671 -0.0175  0.1319  0.1826  0.2275   
vegetacion_densa        230.0  0.6943  0.0611  0.6005  0.6483  0.6848  0.7385   

                           max  
clase                           
contaminacion_alta_NO2  0.8067  
contaminacion_alta_SO2  0.8634  
ozono_anomalo           0.8738  
suelo_urbano            0.2983  
vegetacion_densa        0.8696  

ndbi
                        count    mean     std     min     25%     50%     75%  \
clase                                                                           
contaminaci

In [8]:
print("Stats tiles por banda")
for i, b in enumerate(bands):
    arr = tiles[:, i, :, :]
    print(f"{b:<4} mean={arr.mean():.4f} std={arr.std():.4f} "
          f"min={arr.min():.4f} max={arr.max():.4f} "
          f"zeros={((arr==0).sum()/arr.size*100):.2f}%")

Stats tiles por banda
B1   mean=864.2266 std=912.4138 min=0.0000 max=13626.0000 zeros=0.22%
B2   mean=906.4669 std=949.3516 min=0.0000 max=19944.0000 zeros=0.22%
B3   mean=1077.9440 std=880.1616 min=0.0000 max=19000.0000 zeros=0.22%
B4   mean=985.2714 std=925.5620 min=0.0000 max=18648.0000 zeros=0.23%
B5   mean=1373.0154 std=868.0828 min=0.0000 max=17082.0000 zeros=0.22%
B6   mean=2411.1577 std=852.4259 min=0.0000 max=16591.0000 zeros=0.22%
B7   mean=2855.8225 std=1002.0649 min=0.0000 max=16363.0000 zeros=0.22%
B8   mean=2810.9182 std=1018.5317 min=0.0000 max=16888.0000 zeros=0.22%
B8A  mean=3077.1477 std=1057.5471 min=0.0000 max=16508.0000 zeros=0.22%
B9   mean=3053.1060 std=1170.3116 min=0.0000 max=16189.0000 zeros=0.22%
B11  mean=2160.8337 std=877.9880 min=0.0000 max=15575.0000 zeros=0.22%
B12  mean=1497.4562 std=950.5408 min=0.0000 max=15380.0000 zeros=0.22%
SCL  mean=4.6265 std=1.3190 min=0.0000 max=11.0000 zeros=0.22%


In [9]:
print("Nulos por clase")
nulls = meta.groupby("clase").apply(lambda x: x.isna().mean())
print(nulls.loc[:, nulls.max() > 0].round(3))
print()
print("Outliers por IQR")
for col in ["ndvi", "ndbi", "scl_pct", "no2", "so2", "o3"]:
    q1, q3 = meta[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n = ((meta[col] < lo) | (meta[col] > hi)).sum()
    print(f"{col:<8} lo={lo:.4f} hi={hi:.4f} outliers={n} ({n/len(meta)*100:.1f}%)")

Nulos por clase
                          no2     o3  modis_AOD_047  modis_AOD_055
clase                                                             
contaminacion_alta_NO2  0.000  0.004          0.052          0.052
contaminacion_alta_SO2  0.022  0.000          0.017          0.017
ozono_anomalo           0.304  0.000          0.113          0.113
suelo_urbano            0.152  0.065          0.091          0.091
vegetacion_densa        0.378  0.200          0.091          0.091

Outliers por IQR
ndvi     lo=-0.3300 hi=1.2942 outliers=0 (0.0%)
ndbi     lo=-0.7227 hi=0.4996 outliers=0 (0.0%)
scl_pct  lo=0.5259 hi=1.2845 outliers=105 (9.1%)
no2      lo=-0.0000 hi=0.0001 outliers=34 (3.0%)
so2      lo=-0.0002 hi=0.0004 outliers=140 (12.2%)
o3       lo=0.0992 hi=0.1393 outliers=0 (0.0%)


In [10]:
scl_idx = list(bands).index("SCL")
scl_tiles = tiles[:, scl_idx]
scl_vals = np.unique(scl_tiles)
print("SCL por clase (promedio de pixeles)")
for c in meta["clase"].unique():
    idx = meta["clase"] == c
    sub = scl_tiles[idx.values]
    pix = sub.reshape(idx.sum(), -1)
    print(f"\n{c}")
    for v in scl_vals:
        pct = (pix == v).mean() * 100
        if pct > 0.5:
            print(f"  SCL {int(v):>2}: {pct:.1f}%")

SCL por clase (promedio de pixeles)

contaminacion_alta_NO2
  SCL  0: 0.6%
  SCL  3: 3.3%
  SCL  4: 58.6%
  SCL  5: 23.9%
  SCL  7: 2.6%
  SCL  8: 5.2%
  SCL  9: 4.3%
  SCL 10: 1.0%

vegetacion_densa
  SCL  3: 7.5%
  SCL  4: 83.4%
  SCL  5: 4.2%
  SCL  7: 0.9%
  SCL  8: 2.5%
  SCL  9: 0.7%
  SCL 10: 0.8%

ozono_anomalo
  SCL  3: 5.4%
  SCL  4: 60.8%
  SCL  5: 20.7%
  SCL  7: 2.3%
  SCL  8: 6.3%
  SCL  9: 3.4%

contaminacion_alta_SO2
  SCL  3: 3.8%
  SCL  4: 75.2%
  SCL  5: 12.4%
  SCL  7: 2.2%
  SCL  8: 3.7%
  SCL  9: 2.3%

suelo_urbano
  SCL  3: 2.8%
  SCL  4: 8.9%
  SCL  5: 78.4%
  SCL  7: 2.6%
  SCL  8: 3.3%
  SCL  9: 3.2%


In [11]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
X = tiles.mean(axis=(2, 3))
Xs = StandardScaler().fit_transform(X)
pca = PCA().fit(Xs)
ev = pca.explained_variance_ratio_
print("PCA sobre medias espectrales (13 bandas)")
for i, v in enumerate(ev):
    print(f"  PC{i+1:>2}: {v*100:.2f}%  (acum: {ev[:i+1].sum()*100:.2f}%)")

PCA sobre medias espectrales (13 bandas)
  PC 1: 56.27%  (acum: 56.27%)
  PC 2: 36.70%  (acum: 92.97%)
  PC 3: 4.24%  (acum: 97.21%)
  PC 4: 1.56%  (acum: 98.76%)
  PC 5: 0.60%  (acum: 99.36%)
  PC 6: 0.29%  (acum: 99.65%)
  PC 7: 0.16%  (acum: 99.81%)
  PC 8: 0.11%  (acum: 99.92%)
  PC 9: 0.04%  (acum: 99.96%)
  PC10: 0.02%  (acum: 99.98%)
  PC11: 0.01%  (acum: 99.99%)
  PC12: 0.01%  (acum: 100.00%)
  PC13: 0.00%  (acum: 100.00%)


In [12]:
null_cols = ["no2", "o3", "modis_AOD_047", "modis_AOD_055"]
nulls = meta[null_cols].isna()
print("Patrones de missing")
pat = {}
for row in nulls.itertuples(index=False):
    key = tuple(row)
    pat[key] = pat.get(key, 0) + 1
for p, cnt in sorted(pat.items(), key=lambda x: -x[1]):
    cols = [c for c, v in zip(null_cols, p) if v]
    print(f"  faltan {cols if cols else 'NINGUNO'}: {cnt} ({cnt/len(meta)*100:.1f}%)")

Patrones de missing
  faltan NINGUNO: 895 (77.8%)
  faltan ['no2']: 120 (10.4%)
  faltan ['modis_AOD_047', 'modis_AOD_055']: 48 (4.2%)
  faltan ['no2', 'o3']: 43 (3.7%)
  faltan ['no2', 'modis_AOD_047', 'modis_AOD_055']: 25 (2.2%)
  faltan ['no2', 'o3', 'modis_AOD_047', 'modis_AOD_055']: 9 (0.8%)
  faltan ['o3']: 8 (0.7%)
  faltan ['o3', 'modis_AOD_047', 'modis_AOD_055']: 2 (0.2%)


In [13]:
print("Desvio intra-tile promedio")
for i, b in enumerate(bands[:12]):
    arr = tiles[:, i]
    stds = arr.std(axis=(1, 2))
    print(f"\n{b}")
    for c in meta["clase"].unique():
        idx = meta["clase"] == c
        print(f"  {c}: mean_std={stds[idx.values].mean():.4f}")

Desvio intra-tile promedio

B1
  contaminacion_alta_NO2: mean_std=537.3898
  vegetacion_densa: mean_std=255.8776
  ozono_anomalo: mean_std=481.8808
  contaminacion_alta_SO2: mean_std=331.2340
  suelo_urbano: mean_std=427.5892

B2
  contaminacion_alta_NO2: mean_std=618.4679
  vegetacion_densa: mean_std=319.5411
  ozono_anomalo: mean_std=575.7336
  contaminacion_alta_SO2: mean_std=403.3440
  suelo_urbano: mean_std=693.1958

B3
  contaminacion_alta_NO2: mean_std=596.3360
  vegetacion_densa: mean_std=322.7109
  ozono_anomalo: mean_std=552.8195
  contaminacion_alta_SO2: mean_std=396.0796
  suelo_urbano: mean_std=678.9960

B4
  contaminacion_alta_NO2: mean_std=652.2435
  vegetacion_densa: mean_std=359.1920
  ozono_anomalo: mean_std=617.4697
  contaminacion_alta_SO2: mean_std=446.2459
  suelo_urbano: mean_std=721.0667

B5
  contaminacion_alta_NO2: mean_std=620.3906
  vegetacion_densa: mean_std=365.1635
  ozono_anomalo: mean_std=583.8611
  contaminacion_alta_SO2: mean_std=432.1969
  suelo_urba

In [14]:
from scipy.stats import f_oneway
print("ANOVA F-stat vs clase")
for col in ["ndvi", "ndbi", "scl_pct", "no2", "so2", "o3", "modis_AOD_047", "modis_AOD_055"]:
    groups = [g.dropna().values for _, g in meta.groupby("clase")[col]]
    if all(len(g) > 1 for g in groups):
        f, p = f_oneway(*groups)
        print(f"  {col:<14} F={f:>8.2f}  p={p:.2e}")
    else:
        print(f"  {col:<14} insuficientes datos")

ANOVA F-stat vs clase
  ndvi           F=  475.34  p=1.80e-241
  ndbi           F=  416.46  p=1.73e-221
  scl_pct        F=    4.83  p=7.27e-04
  no2            F=  180.23  p=7.67e-115
  so2            F=  249.43  p=4.14e-154
  o3             F=  131.86  p=8.70e-92
  modis_AOD_047  F=   10.06  p=5.37e-08
  modis_AOD_055  F=   10.06  p=5.39e-08
